# Live demo: SKG-IF API for ORKG

This notebook demonstrates how RAMOSE turns a SPARQL triplestore into a [SKG-IF](https://skg-if.github.io/interoperability-framework/) compliant REST API.
The target triplestore is [ORKG](https://orkg.org/) (Open Research Knowledge Graph).

To run it interactively, click **Open in Colab** or **Launch Binder** in the top bar.

In [1]:
!pip install -q ramose

In [2]:
from urllib.request import urlretrieve

urlretrieve(
    "https://raw.githubusercontent.com/opencitations/ramose/master/test/data/orkg_skgif.hf",
    "orkg_skgif.hf",
)
print("Downloaded orkg_skgif.hf")

Downloaded orkg_skgif.hf


## Query a single research product

Load the spec file, resolve an operation, and execute it against ORKG's live SPARQL endpoint.

In [3]:
import json

from ramose import APIManager, Operation

am = APIManager(["orkg_skgif.hf"])
op = am.get_op("/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R25543")

if isinstance(op, Operation):
    status, body, content_type, headers = op.exec()
    print(json.dumps(json.loads(body), indent=2))

{
  "@context": [
    "https://w3id.org/skg-if/context/1.1.0/skg-if.json",
    "https://w3id.org/skg-if/context/1.0.0/skg-if-api.json",
    {
      "@base": "https://w3id.org/skg-if/sandbox/"
    }
  ],
  "meta": {
    "local_identifier": "/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R25543?page=1&page_size=1",
    "entity_type": "search_result_page",
    "part_of": {
      "local_identifier": "/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R25543",
      "entity_type": "search_result",
      "total_items": 1,
      "first_page": {
        "local_identifier": "/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R25543?page=1&page_size=1",
        "entity_type": "search_result_page"
      },
      "last_page": {
        "local_identifier": "/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R25543?page=1&page_size=1",
        "entity_type": "search_result_page"
      }
    }
  },
  "@graph": [
    {
      "local_identifier": "http://orkg.org/orkg/resource/R25543",
 

The response is a JSON-LD document following the [SKG-IF specification](https://skg-if.github.io/interoperability-framework/docs/research-product.html). It includes:

- **Pagination metadata** (`meta`): page info, total items, navigation links
- **Product data** (`@graph`): title, identifiers (DOI), contributions (ordered authors), topics, manifestation dates, and venue

All of this is generated from a single `.hf` spec file and the ORKG SPARQL endpoint. No custom backend code is needed.

## Generated HTML documentation

RAMOSE also generates interactive API documentation from the same spec file.

In [4]:
import html
import warnings

from IPython.display import HTML

from ramose import HTMLDocumentationHandler

warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")

doc_handler = HTMLDocumentationHandler(am)
_, html_content = doc_handler.get_documentation()

HTML(
    f'<iframe srcdoc="{html.escape(html_content)}" width="100%" height="600" style="border:1px solid #ccc; border-radius:4px"></iframe>'
)

## Next steps

To adapt this for a different triplestore:

1. Write a `.hf` spec file mapping your SPARQL schema to the [product column schema](https://opencitations.github.io/ramose/08-skgif.html)
2. Point `#endpoint` to your SPARQL endpoint
3. Run `ramose -s your_spec.hf -w 127.0.0.1:8080` or use the Python API as shown above

Full documentation: [opencitations.github.io/ramose](https://opencitations.github.io/ramose/)